# What Are Agents — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicai/learn-notebooks/blob/main/notebooks/ai-agents/6-1/what-are-agents.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicai/learn-notebooks/blob/main/notebooks/ai-agents/6-1/what-are-agents.ipynb)

> Companion to the **What Are Agents** article. Build a tiny restaurant recommender **two ways** —
> first as a **workflow** (your code decides the order), then as an **agent** (the model decides) —
> and *watch* the difference happen, line by line.

## What you'll build

1. Two tiny **tools**: `extract_prefs` and `search_restaurants`.
2. A **workflow** version — a fixed pipeline you control.
3. An **agent** version — a loop where the *model* chooses each step.

**No API key needed.** To see the mechanics clearly and deterministically, we use a tiny
**mock "model"** (a scripted policy). The **last section** shows the *real* Claude version — that
one runs in Colab once you add your `ANTHROPIC_API_KEY`.

### Setup (nothing to install for the mock sections)

In [1]:
# Only needed for the FINAL "real Claude" section — safe to skip for now:
# %pip install -q anthropic
import os, json
print("ready — no API key required for the mock sections")

ready — no API key required for the mock sections


## The two tools

A **tool** is just a typed function the model may call. Ours return canned data so the notebook
is self-contained.

In [2]:
def extract_prefs(message: str) -> dict:
    """Pretend-NLU: pull structured preferences out of a free-text message."""
    return {"food": "spicy", "city": "Boston", "avoid": "shellfish"}

def search_restaurants(query: str, location: str) -> list:
    """Pretend restaurant search near a location."""
    return [
        {"name": "Chili House",   "tags": ["spicy"]},
        {"name": "Pepper Bowl",   "tags": ["spicy", "vegetarian"]},
        {"name": "Ocean Grill",   "tags": ["seafood", "shellfish"]},
    ]

# tools the AGENT is allowed to call, keyed by name
TOOLS = {"extract_prefs": extract_prefs, "search_restaurants": search_restaurants}
print("tools:", list(TOOLS))
print("sample extract_prefs:", extract_prefs("spicy dinner in Boston, no shellfish"))

tools: ['extract_prefs', 'search_restaurants']
sample extract_prefs: {'food': 'spicy', 'city': 'Boston', 'avoid': 'shellfish'}


## 1. Workflow — **your code** decides the order

Fixed pipeline: `extract → search → pick`. Always. The arrows are hand-written; the "model"
only fills in each box. Predictable, cheap, trivially testable.

In [3]:
def pick(places, prefs):
    """The 'model' box: choose a place honoring the avoid-list. (Deterministic stand-in.)"""
    safe = [p for p in places if prefs["avoid"] not in p["tags"]]   # drop the shellfish place
    return safe[0]["name"] if safe else "no safe match"

def recommend_workflow(message: str):
    print("WORKFLOW — I (the code) run a FIXED order:")
    prefs  = extract_prefs(message);            print("  step 1  extract_prefs ->", prefs)
    places = search_restaurants(prefs["food"], prefs["city"]); print("  step 2  search_restaurants ->", [p["name"] for p in places])
    choice = pick(places, prefs);               print("  step 3  pick ->", choice)
    return choice

result = recommend_workflow("spicy dinner near Boston, no shellfish")
print("\nfinal:", result)
print("\nWHO DECIDED THE ORDER? -> my Python did. extract, then search, then pick. Every time.")

WORKFLOW — I (the code) run a FIXED order:
  step 1  extract_prefs -> {'food': 'spicy', 'city': 'Boston', 'avoid': 'shellfish'}
  step 2  search_restaurants -> ['Chili House', 'Pepper Bowl', 'Ocean Grill']
  step 3  pick -> Chili House

final: Chili House

WHO DECIDED THE ORDER? -> my Python did. extract, then search, then pick. Every time.


## 2. Agent — the **model** decides the order (a loop)

Now we hand the model a **goal** and the **tools**, and let *it* choose each step inside a loop.
Our mock "model" is a tiny scripted policy so you can watch the mechanics; in real life this is the LLM.

Watch three things print each turn: **who decided**, the **tool call**, and the growing **memory**.

In [4]:
def mock_model(goal: str, messages: list):
    """A stand-in for the LLM. Chooses the next action from what it has seen so far.
    Real LLMs output this same shape (a tool call, or a final answer)."""
    n_obs = sum(1 for m in messages if m["role"] == "tool")   # how many observations fed back
    if n_obs == 0:
        return {"kind": "tool_call", "name": "extract_prefs",     "args": {"message": goal}}
    if n_obs == 1:
        return {"kind": "tool_call", "name": "search_restaurants", "args": {"query": "spicy", "location": "Boston"}}
    return {"kind": "final", "text": "I'd recommend **Chili House** — spicy, in Boston, and shellfish-free."}

def run_agent(goal: str, max_steps: int = 6):
    messages = [{"role": "user", "content": goal}]     # <-- MEMORY starts here
    for step in range(1, max_steps + 1):
        action = mock_model(goal, messages)            # <-- THE MODEL decides the next step
        if action["kind"] == "final":
            print(f"step {step}: 🧠 model decided -> DONE (stop)")
            return action["text"]
        name, args = action["name"], action["args"]
        print(f"step {step}: 🧠 model decided -> call `{name}({args})`")
        observation = TOOLS[name](**args)              # <-- YOUR code runs the tool
        print(f"          🔧 tool ran -> {observation if name!='search_restaurants' else [p['name'] for p in observation]}")
        messages.append({"role": "assistant", "content": f"call {name}"})
        messages.append({"role": "tool", "content": observation})   # <-- observation fed back into MEMORY
        print(f"          🧠 memory now has {len(messages)} messages")
    return "stopped: hit max_steps cap"

answer = run_agent("Recommend a restaurant for: spicy dinner near Boston, no shellfish")
print("\nfinal:", answer)

step 1: 🧠 model decided -> call `extract_prefs({'message': 'Recommend a restaurant for: spicy dinner near Boston, no shellfish'})`
          🔧 tool ran -> {'food': 'spicy', 'city': 'Boston', 'avoid': 'shellfish'}
          🧠 memory now has 3 messages
step 2: 🧠 model decided -> call `search_restaurants({'query': 'spicy', 'location': 'Boston'})`
          🔧 tool ran -> ['Chili House', 'Pepper Bowl', 'Ocean Grill']
          🧠 memory now has 5 messages
step 3: 🧠 model decided -> DONE (stop)

final: I'd recommend **Chili House** — spicy, in Boston, and shellfish-free.


## Watch the difference

Same task, same tools — the **only** difference is *who sequenced the steps*.

| | Workflow | Agent |
|---|---|---|
| Who chose the order | **your code** (fixed) | **the model** (each loop) |
| Steps | always 3 | a variable it decides |
| Stop condition | end of function | **model** says "done" |

**The one interview test:** *at runtime, who decides the next step — my code, or the model?*
Code ⇒ workflow. Model ⇒ agent.

## 3. The real thing (Claude) — run this in **Colab** with your key

Identical loop; the mock model is swapped for a real `client.messages.create` call. It runs when
`ANTHROPIC_API_KEY` is set (Colab → *Secrets*). With no key it just prints a friendly note, so the
notebook still executes top-to-bottom.

In [5]:
def run_agent_claude(goal: str, max_steps: int = 6):
    if not os.getenv("ANTHROPIC_API_KEY"):
        print("(no ANTHROPIC_API_KEY set — add it in Colab Secrets to run the real Claude loop)")
        return None
    from anthropic import Anthropic
    client = Anthropic()
    tool_specs = [
        {"name": "extract_prefs", "description": "Extract food, city, avoid-list from a message",
         "input_schema": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
        {"name": "search_restaurants", "description": "Find restaurants for a food query near a location",
         "input_schema": {"type": "object",
             "properties": {"query": {"type": "string"}, "location": {"type": "string"}}, "required": ["query", "location"]}},
    ]
    messages = [{"role": "user", "content": goal}]
    for step in range(1, max_steps + 1):
        resp = client.messages.create(model="claude-sonnet-4-5", max_tokens=1024,
            system="You are a food-recommendation agent. Use tools, then answer.",
            tools=tool_specs, messages=messages)
        print(f"step {step}: stop_reason = {resp.stop_reason}")   # <-- watch WHO decides "done"
        if resp.stop_reason != "tool_use":
            return resp.content[0].text
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for block in resp.content:
            if block.type == "tool_use":
                out = TOOLS[block.name](**block.input)            # YOUR code runs the tool
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(out)})
        messages.append({"role": "user", "content": results})     # feed observations back
    return "stopped: hit max_steps cap"

print(run_agent_claude("Recommend a restaurant for: spicy dinner near Boston, no shellfish"))

(no ANTHROPIC_API_KEY set — add it in Colab Secrets to run the real Claude loop)
None


## Recap

- **Agent = LLM + tools + loop + memory.** The loop is what makes it an *agent*.
- **Workflow vs agent** = *who decides the next step at runtime* (your code vs the model).
- You watched both run the **same** restaurant task; only the sequencing differed.

← Back to the **What Are Agents** article for the full explanation, diagrams, and interview Q&A.